# NLP-04 — 08_error_analysis

Publication copy of the research notebook.

Saved outputs and unnecessary metadata have been removed. Private annotations, raw datasets and model checkpoints are retained separately.

Some cells require external research artifacts. The public notebook is not a standalone reproduction package.

In [ ]:
from google.colab import drive, files
from pathlib import Path

drive.mount('/content/drive')

SRC_DIR = Path(
    '/content/drive/MyDrive/'
    'NLP04_Toxic_Comment_Classification/03_src'
)

script_path = SRC_DIR / '08_error_analysis_cell1.py'

if not script_path.exists():
    uploaded = files.upload()

    assert len(uploaded) == 1

    filename, content = next(iter(uploaded.items()))

    assert filename.startswith('08_error_analysis_cell1')
    assert filename.endswith('.py')

    compile(content, filename, 'exec')
    script_path.write_bytes(content)

exec(
    compile(
        script_path.read_text(),
        str(script_path),
        'exec'
    ),
    globals()
)

In [ ]:
# Cell 2: Label-level and paired error analysis

error_rows = []
paired_rows = []

prediction_frames = {
    PRIMARY_RUN_ID: primary_predictions,
    BASELINE_RUN_ID: baseline_predictions
}

predictions = {}

# Calculate label-level errors for both frozen models
for run_id, frame in prediction_frames.items():

    pred = frame[
        [f'pred_optimized_{label}' for label in LABELS]
    ].to_numpy(dtype=bool)

    predictions[run_id] = pred

    for j, label in enumerate(LABELS):

        actual = y_test[:, j].astype(bool)
        predicted = pred[:, j]

        tp = int((actual & predicted).sum())
        tn = int((~actual & ~predicted).sum())
        fp = int((~actual & predicted).sum())
        fn = int((actual & ~predicted).sum())

        assert tp + tn + fp + fn == len(test_df)

        error_rows.append({
            'Run ID': run_id,
            'Label': label,
            'TP': tp,
            'TN': tn,
            'FP': fp,
            'FN': fn,
            'Total Errors': fp + fn,
            'Precision': tp / (tp + fp) if tp + fp else 0,
            'Recall': tp / (tp + fn),
            'FPR (%)': 100 * fp / (fp + tn),
            'FNR (%)': 100 * fn / (fn + tp),
            'FP Share of Errors (%)': (
                100 * fp / (fp + fn) if fp + fn else 0
            )
        })

error_summary = pd.DataFrame(error_rows)

# Verify counts against the official evaluation
reference = per_label_test[
    (per_label_test['Run ID'].isin(prediction_frames)) &
    (per_label_test['Operating Point'] == 'optimized')
]

verified = error_summary.merge(
    reference,
    on=['Run ID', 'Label'],
    suffixes=('_new', '_saved'),
    validate='one_to_one'
)

assert len(verified) == 12

for metric in ['TP', 'TN', 'FP', 'FN']:
    np.testing.assert_array_equal(
        verified[f'{metric}_new'],
        verified[f'{metric}_saved']
    )

# Compare errors on exactly the same comments
primary_pred = predictions[PRIMARY_RUN_ID]
baseline_pred = predictions[BASELINE_RUN_ID]

for j, label in enumerate(LABELS):

    actual = y_test[:, j].astype(bool)

    for outcome in ['FP', 'FN']:

        eligible = ~actual if outcome == 'FP' else actual

        primary_error = (
            primary_pred[:, j] != actual
        ) & eligible

        baseline_error = (
            baseline_pred[:, j] != actual
        ) & eligible

        shared = int((primary_error & baseline_error).sum())

        corrected = int(
            (~primary_error & baseline_error).sum()
        )

        introduced = int(
            (primary_error & ~baseline_error).sum()
        )

        neither = int(
            (eligible & ~primary_error & ~baseline_error).sum()
        )

        assert (
            shared + corrected + introduced + neither
            == eligible.sum()
        )

        paired_rows.append({
            'Label': label,
            'Error Type': outcome,
            'Eligible Comments': int(eligible.sum()),
            'Shared Errors': shared,
            'Corrected by RoBERTa': corrected,
            'Introduced by RoBERTa': introduced,
            'Neither Model Wrong': neither,
            'Net Error Reduction': corrected - introduced
        })

paired_errors = pd.DataFrame(paired_rows)

assert len(error_summary) == 12
assert len(paired_errors) == 12

# Save aggregate results
save_csv_atomic(
    TABLES_DIR / 'label_error_summary.csv',
    error_summary
)

save_csv_atomic(
    TABLES_DIR / 'paired_model_error_comparison.csv',
    paired_errors
)

# Display primary model errors
print('\nPrimary RoBERTa model error summary')

display(
    error_summary[
        error_summary['Run ID'] == PRIMARY_RUN_ID
    ].drop(columns='Run ID').round(4)
)

print('\nPaired comparison against Logistic Regression')

display(paired_errors)

# Visualize false-positive and false-negative rates
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(LABELS))
width = 0.36

for ax, metric in zip(axes, ['FPR (%)', 'FNR (%)']):

    for offset, run_id, name, color in [
        (-width / 2, PRIMARY_RUN_ID, 'RoBERTa', '#326796'),
        (width / 2, BASELINE_RUN_ID, 'Logistic Regression', '#E09B45')
    ]:

        values = error_summary[
            error_summary['Run ID'] == run_id
        ].set_index('Label').loc[LABELS, metric]

        ax.bar(
            x + offset, values,
            width=width,
            label=name,
            color=color
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [label.replace('_', '\n') for label in LABELS]
    )
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)
    ax.legend()

fig.suptitle('Official test: label-level error comparison')

plt.tight_layout()

figure_path = FIGURES_DIR / 'F01_label_error_comparison.png'

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

# Verify that the frozen protocol is unchanged
assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

print('\nCell 2 summary')
print('Models analyzed:', len(predictions))
print('Label-level records:', len(error_summary))
print('Paired comparison records:', len(paired_errors))
print('Confusion counts: Verified')
print('Tables saved: 2')
print('Figures saved: 1')
print('Additional training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 2 completed successfully.')

In [ ]:
# Cell 3: Validation-to-test generalization analysis

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

def partition_metrics(frame, actual):

    scores = frame[
        [f'score_{label}' for label in LABELS]
    ].to_numpy(dtype=float)

    pred = frame[
        [f'pred_optimized_{label}' for label in LABELS]
    ].to_numpy(dtype=bool)

    actual = actual.astype(bool)
    rows = []

    for j, label in enumerate(LABELS):

        yt = actual[:, j]
        yp = pred[:, j]

        tp = int((yt & yp).sum())
        tn = int((~yt & ~yp).sum())
        fp = int((~yt & yp).sum())
        fn = int((yt & ~yp).sum())

        precision = tp / (tp + fp) if tp + fp else 0
        recall = tp / (tp + fn)
        f1 = 2 * tp / (2 * tp + fp + fn)

        rows.append({
            'Label': label,
            'Samples': len(yt),
            'Positive Support': tp + fn,
            'Prevalence': yt.mean(),
            'Precision': precision,
            'Recall': recall,
            'F1': f1,
            'AP': average_precision_score(yt, scores[:, j]),
            'ROC-AUC': roc_auc_score(yt, scores[:, j]),
            'FPR': fp / (fp + tn),
            'FNR': fn / (fn + tp),
            'TP': tp,
            'TN': tn,
            'FP': fp,
            'FN': fn
        })

    return pd.DataFrame(rows)


validation_metrics = partition_metrics(
    primary_validation_predictions,
    y_validation
)

test_metrics = partition_metrics(
    primary_predictions,
    y_test
)

# Verify reconstructed metrics
entry = FROZEN_MODELS[PRIMARY_RUN_ID]

np.testing.assert_allclose(
    validation_metrics['AP'].mean(),
    entry['validation_macro_ap'],
    atol=1e-12,
    rtol=0
)

np.testing.assert_allclose(
    validation_metrics['F1'].mean(),
    entry['validation_optimized_macro_f1'],
    atol=1e-12,
    rtol=0
)

np.testing.assert_allclose(
    test_metrics['AP'].mean(),
    evaluation_completion['primary_test_macro_ap'],
    atol=1e-12,
    rtol=0
)

np.testing.assert_allclose(
    test_metrics['F1'].mean(),
    evaluation_completion[
        'primary_test_optimized_macro_f1'
    ],
    atol=1e-12,
    rtol=0
)

# Compare the same frozen model across partitions
generalization = validation_metrics.merge(
    test_metrics,
    on='Label',
    suffixes=('_Validation', '_Test'),
    validate='one_to_one'
)

comparison_metrics = [
    'Prevalence', 'Precision', 'Recall',
    'F1', 'AP', 'ROC-AUC', 'FPR', 'FNR'
]

for metric in comparison_metrics:

    generalization[f'{metric} Change (pp)'] = 100 * (
        generalization[f'{metric}_Test'] -
        generalization[f'{metric}_Validation']
    )

assert len(generalization) == 6

# Descriptive prevalence standardization
def precision_at_prevalence(prevalence, recall, fpr):

    numerator = prevalence * recall

    denominator = (
        numerator + (1 - prevalence) * fpr
    )

    return numerator / denominator if denominator else 0


def f1_from_precision_recall(precision, recall):

    return (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )


decomposition_rows = []

for _, row in generalization.iterrows():

    val_prev = row['Prevalence_Validation']
    test_prev = row['Prevalence_Test']

    val_recall = row['Recall_Validation']
    val_fpr = row['FPR_Validation']

    # Change prevalence while holding validation rates fixed
    expected_precision = precision_at_prevalence(
        test_prev,
        val_recall,
        val_fpr
    )

    expected_f1 = f1_from_precision_recall(
        expected_precision,
        val_recall
    )

    for metric, expected in [
        ('Precision', expected_precision),
        ('F1', expected_f1)
    ]:

        val_value = row[f'{metric}_Validation']
        test_value = row[f'{metric}_Test']

        prevalence_effect = expected - val_value
        rate_effect = test_value - expected
        total_change = test_value - val_value

        np.testing.assert_allclose(
            prevalence_effect + rate_effect,
            total_change,
            atol=1e-12,
            rtol=0
        )

        decomposition_rows.append({
            'Label': row['Label'],
            'Metric': metric,
            'Validation': val_value,
            'Prevalence-adjusted Validation': expected,
            'Actual Test': test_value,
            'Prevalence Component (pp)': (
                100 * prevalence_effect
            ),
            'Error-rate Component (pp)': (
                100 * rate_effect
            ),
            'Total Change (pp)': 100 * total_change
        })

gap_decomposition = pd.DataFrame(decomposition_rows)

# Summarize aggregate changes
macro_rows = []

for metric in [
    'AP', 'F1', 'Precision', 'Recall',
    'ROC-AUC', 'FPR', 'FNR'
]:

    val = validation_metrics[metric].mean()
    test = test_metrics[metric].mean()

    macro_rows.append({
        'Metric': f'Macro {metric}',
        'Validation': val,
        'Test': test,
        'Change (pp)': 100 * (test - val)
    })

macro_gap = pd.DataFrame(macro_rows)

assert len(gap_decomposition) == 12
assert len(macro_gap) == 7

# Save results
save_csv_atomic(
    TABLES_DIR / 'validation_test_label_comparison.csv',
    generalization
)

save_csv_atomic(
    TABLES_DIR / 'validation_test_macro_comparison.csv',
    macro_gap
)

save_csv_atomic(
    TABLES_DIR / 'prevalence_gap_decomposition.csv',
    gap_decomposition
)

# Display important findings
print('\nAggregate validation-to-test performance gap')
display(macro_gap.round(6))

print('\nPer-label performance changes')

display(
    generalization[[
        'Label',
        'Prevalence Change (pp)',
        'Precision Change (pp)',
        'Recall Change (pp)',
        'F1 Change (pp)',
        'AP Change (pp)',
        'FPR Change (pp)',
        'FNR Change (pp)'
    ]].round(4)
)

print('\nDescriptive decomposition of precision and F1 changes')
display(gap_decomposition.round(4))

# Visualize ranking and classification performance
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(LABELS))
width = 0.36

for ax, metric in zip(axes, ['AP', 'F1']):

    ax.bar(
        x - width / 2,
        validation_metrics[metric],
        width,
        label='Validation',
        color='#9DB3C9'
    )

    ax.bar(
        x + width / 2,
        test_metrics[metric],
        width,
        label='Official test',
        color='#326796'
    )

    ax.set_xticks(x)
    ax.set_xticklabels([
        label.replace('_', '\n')
        for label in LABELS
    ])

    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title(metric)
    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)
    ax.legend()

fig.suptitle(
    'Frozen primary model: validation-to-test performance'
)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / 'F02_validation_test_performance_gap.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

# Confirm the frozen protocol remains unchanged
assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

print('\nCell 3 summary')
print('Validation samples:', len(y_validation))
print('Official test samples:', len(y_test))
print('Per-label comparison records:', len(generalization))
print('Aggregate comparison records:', len(macro_gap))
print('Decomposition records:', len(gap_decomposition))
print('Reconstructed metrics: Verified')
print('Tables saved: 3')
print('Figures saved: 1')
print('Additional inference and training: None')
print('Frozen protocol: Unchanged')
print('Cell 3 completed successfully.')

In [ ]:
# Cell 4: Text length and truncation analysis

from transformers import AutoTokenizer
from sklearn.metrics import average_precision_score

# Restore the exact frozen tokenizer
entry = FROZEN_MODELS[PRIMARY_RUN_ID]
max_length = int(entry['maximum_length'])

assert max_length == 256

model_dir = PROJECT_ROOT / entry['model_path']

for record in entry['model_files']:
    if Path(record['path']).name.startswith('tokenizer'):
        verify_file(record)

tokenizer = AutoTokenizer.from_pretrained(
    model_dir,
    local_files_only=True,
    use_fast=True
)

assert tokenizer.is_fast
assert tokenizer.truncation_side == 'right'

# Calculate token lengths before truncation
def get_full_token_lengths(frame, partition):

    lengths = []

    for start in range(0, len(frame), 256):

        texts = frame['comment_text'].iloc[
            start:start + 256
        ].tolist()

        encoded = tokenizer(
            texts,
            truncation=False,
            padding=False,
            add_special_tokens=True,
            return_attention_mask=False,
            return_token_type_ids=False,
            verbose=False
        )

        lengths.extend(
            len(ids) for ids in encoded['input_ids']
        )

    lengths = np.asarray(lengths, dtype=np.int32)

    assert len(lengths) == len(frame)
    assert (lengths >= 2).all()

    print(
        partition,
        '| Comments:', len(lengths),
        '| Truncated:', int((lengths > max_length).sum())
    )

    return lengths


validation_lengths = get_full_token_lengths(
    validation_df, 'Validation'
)

test_lengths = get_full_token_lengths(
    test_df, 'Official test'
)

# Verify against the original test tokenization cache
tokenization_path = verify_evaluation_artifact(
    EVALUATION_RESULTS_DIR /
    'configs/transformer_test_tokenization.json'
)

tokenization_config = read_json(tokenization_path)

cache_record = next(
    record for record in tokenization_config['caches']
    if PRIMARY_RUN_ID in record['run_ids']
)

cache_path = PROJECT_ROOT / cache_record['cache_path']

assert calculate_sha256(cache_path) == (
    cache_record['cache_sha256']
)

cached_tokens = pd.read_parquet(
    cache_path,
    columns=['id', 'input_ids']
)

np.testing.assert_array_equal(
    cached_tokens['id'].astype(str).to_numpy(),
    test_ids
)

cached_lengths = cached_tokens['input_ids'].map(
    len
).to_numpy()

np.testing.assert_array_equal(
    np.minimum(test_lengths, max_length),
    cached_lengths
)

print('Original tokenization cache: Verified')

del cached_tokens
gc.collect()

# Define fixed length groups
length_groups = [
    ('1–64', 1, 64),
    ('65–128', 65, 128),
    ('129–256', 129, 256),
    ('>256', 257, np.inf)
]

analysis_inputs = [
    (
        'Validation',
        PRIMARY_RUN_ID,
        primary_validation_predictions,
        y_validation,
        validation_lengths
    ),
    (
        'Official test',
        PRIMARY_RUN_ID,
        primary_predictions,
        y_test,
        test_lengths
    ),
    (
        'Official test',
        BASELINE_RUN_ID,
        baseline_predictions,
        y_test,
        test_lengths
    )
]

length_rows = []
label_length_rows = []

# Evaluate saved predictions within each length group
for partition, run_id, frame, actual, lengths in analysis_inputs:

    actual = actual.astype(bool)

    pred = frame[
        [f'pred_optimized_{label}' for label in LABELS]
    ].to_numpy(dtype=bool)

    scores = frame[
        [f'score_{label}' for label in LABELS]
    ].to_numpy(dtype=float)

    for group, lower, upper in length_groups:

        mask = (lengths >= lower) & (lengths <= upper)
        n = int(mask.sum())

        assert n > 0

        yt = actual[mask]
        yp = pred[mask]

        tp = (yt & yp).sum(axis=0)
        tn = (~yt & ~yp).sum(axis=0)
        fp = (~yt & yp).sum(axis=0)
        fn = (yt & ~yp).sum(axis=0)

        assert np.all(tp + tn + fp + fn == n)

        def safe_div(a, b):
            return a / b if b else np.nan

        length_rows.append({
            'Partition': partition,
            'Run ID': run_id,
            'Length Group': group,
            'Comments': n,
            'Comment Share (%)': 100 * n / len(actual),
            'Any Toxic Prevalence (%)': (
                100 * yt.any(axis=1).mean()
            ),
            'FP': int(fp.sum()),
            'FN': int(fn.sum()),
            'Micro Precision': safe_div(
                tp.sum(), tp.sum() + fp.sum()
            ),
            'Micro Recall': safe_div(
                tp.sum(), tp.sum() + fn.sum()
            ),
            'Micro F1': safe_div(
                2 * tp.sum(),
                2 * tp.sum() + fp.sum() + fn.sum()
            ),
            'Micro FPR (%)': 100 * safe_div(
                fp.sum(), fp.sum() + tn.sum()
            ),
            'Micro FNR (%)': 100 * safe_div(
                fn.sum(), fn.sum() + tp.sum()
            ),
            'Hamming Loss': (yt != yp).mean(),
            'Exact Match Accuracy': (
                yt == yp
            ).all(axis=1).mean()
        })

        for j, label in enumerate(LABELS):

            positives = int(tp[j] + fn[j])
            negatives = int(tn[j] + fp[j])

            ap = (
                average_precision_score(
                    yt[:, j], scores[mask, j]
                )
                if positives > 0 and negatives > 0
                else np.nan
            )

            label_length_rows.append({
                'Partition': partition,
                'Run ID': run_id,
                'Length Group': group,
                'Label': label,
                'Comments': n,
                'Positive Support': positives,
                'Negative Support': negatives,
                'TP': int(tp[j]),
                'TN': int(tn[j]),
                'FP': int(fp[j]),
                'FN': int(fn[j]),
                'Precision': safe_div(
                    tp[j], tp[j] + fp[j]
                ),
                'Recall': safe_div(
                    tp[j], positives
                ),
                'F1': safe_div(
                    2 * tp[j],
                    2 * tp[j] + fp[j] + fn[j]
                ),
                'AP': ap,
                'FPR (%)': 100 * safe_div(
                    fp[j], negatives
                ),
                'FNR (%)': 100 * safe_div(
                    fn[j], positives
                )
            })

length_summary = pd.DataFrame(length_rows)

label_length_summary = pd.DataFrame(
    label_length_rows
)

assert len(length_summary) == 12
assert len(label_length_summary) == 72

# Verify that subgroup counts reconstruct all errors
for partition, run_id, frame, actual, lengths in analysis_inputs:

    selected = length_summary[
        (length_summary['Partition'] == partition) &
        (length_summary['Run ID'] == run_id)
    ]

    assert selected['Comments'].sum() == len(actual)

    pred = frame[
        [f'pred_optimized_{label}' for label in LABELS]
    ].to_numpy(dtype=bool)

    actual = actual.astype(bool)

    assert selected['FP'].sum() == (
        (~actual & pred).sum()
    )

    assert selected['FN'].sum() == (
        (actual & ~pred).sum()
    )

# Summarize actual truncation
truncation_rows = []

for partition, lengths in [
    ('Validation', validation_lengths),
    ('Official test', test_lengths)
]:

    truncated = lengths > max_length

    truncation_rows.append({
        'Partition': partition,
        'Comments': len(lengths),
        'Mean Tokens': lengths.mean(),
        'Median Tokens': np.median(lengths),
        'P95 Tokens': np.percentile(lengths, 95),
        'Maximum Tokens': int(lengths.max()),
        'Exactly 256 Tokens': int(
            (lengths == max_length).sum()
        ),
        'Truncated Comments': int(truncated.sum()),
        'Truncated Comments (%)': (
            100 * truncated.mean()
        ),
        'Tokens Removed': int(
            np.maximum(lengths - max_length, 0).sum()
        ),
        'Mean Tokens Removed If Truncated': (
            (lengths[truncated] - max_length).mean()
            if truncated.any() else 0
        )
    })

truncation_summary = pd.DataFrame(truncation_rows)

# Save aggregate results
save_csv_atomic(
    TABLES_DIR / 'text_length_error_summary.csv',
    length_summary
)

save_csv_atomic(
    TABLES_DIR / 'per_label_length_error_summary.csv',
    label_length_summary
)

save_csv_atomic(
    TABLES_DIR / 'actual_truncation_summary.csv',
    truncation_summary
)

print('\nActual truncation summary')
display(truncation_summary.round(4))

print('\nPerformance by comment length')
display(length_summary.round(4))

# Visualize test performance across length groups
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

x = np.arange(len(length_groups))
width = 0.36
group_names = [group[0] for group in length_groups]

for ax, metric in zip(
    axes,
    ['Micro F1', 'Micro FPR (%)', 'Micro FNR (%)']
):

    for offset, run_id, name, color in [
        (
            -width / 2, PRIMARY_RUN_ID,
            'RoBERTa', '#326796'
        ),
        (
            width / 2, BASELINE_RUN_ID,
            'Logistic Regression', '#E09B45'
        )
    ]:

        values = length_summary[
            (length_summary['Partition'] == 'Official test') &
            (length_summary['Run ID'] == run_id)
        ].set_index('Length Group').loc[
            group_names, metric
        ]

        ax.bar(
            x + offset, values,
            width=width,
            color=color,
            label=name
        )

    ax.set_xticks(x)
    ax.set_xticklabels(group_names)
    ax.set_xlabel('Original RoBERTa token length')
    ax.set_title(metric)
    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)

    if metric == 'Micro F1':
        ax.set_ylim(0, 1.05)

    ax.legend()

fig.suptitle(
    'Official test performance by comment length'
)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / 'F03_text_length_error_analysis.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

# Confirm that frozen artifacts remain unchanged
assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

print('\nCell 4 summary')
print('Validation comments:', len(validation_lengths))
print('Test comments:', len(test_lengths))
print('Length groups:', len(length_groups))
print('Aggregate records:', len(length_summary))
print('Per-label records:', len(label_length_summary))
print('Original tokenization cache: Verified')
print('Subgroup error counts: Verified')
print('Tables saved: 3')
print('Figures saved: 1')
print('Additional model training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 4 completed successfully.')

In [ ]:
# Cell 5: Reproducible stratified error sampling

rng = np.random.default_rng(SEED)

primary_pred = primary_predictions[
    [f'pred_optimized_{label}' for label in LABELS]
].to_numpy(dtype=bool)

primary_scores = primary_predictions[
    [f'score_{label}' for label in LABELS]
].to_numpy(dtype=float)

sample_rows = []
sampling_rows = []

# Sample independently within each label and error stratum
for label, quotas in review_quotas.items():

    j = LABELS.index(label)

    actual = y_test[:, j].astype(bool)
    predicted = primary_pred[:, j]

    error_masks = {
        'FP': ~actual & predicted,
        'FN': actual & ~predicted
    }

    for error_type, quota in quotas.items():

        eligible = np.flatnonzero(
            error_masks[error_type]
        )

        sample_size = min(quota, len(eligible))

        selected = rng.choice(
            eligible,
            size=sample_size,
            replace=False
        )

        threshold = FROZEN_MODELS[
            PRIMARY_RUN_ID
        ]['optimized_thresholds'][label]

        for i in selected:

            sample_rows.append({
                'Comment ID': test_ids[i],
                'Row Index': int(i),
                'Target Label': label,
                'Error Type': error_type,
                'Ground Truth': int(actual[i]),
                'Prediction': int(predicted[i]),
                'Score': float(primary_scores[i, j]),
                'Threshold': float(threshold),
                'Original Token Length': int(test_lengths[i]),
                'Truncated': bool(test_lengths[i] > 256)
            })

        sampling_rows.append({
            'Label': label,
            'Error Type': error_type,
            'Available Cases': len(eligible),
            'Requested Cases': quota,
            'Sampled Cases': sample_size
        })

review_sample = pd.DataFrame(sample_rows)

# Randomize review order and assign case identifiers
review_sample = review_sample.iloc[
    rng.permutation(len(review_sample))
].reset_index(drop=True)

review_sample.insert(
    0,
    'Case ID',
    [
        f'ERR_{i+1:03d}'
        for i in range(len(review_sample))
    ]
)

sampling_summary = pd.DataFrame(sampling_rows)

assert len(review_sample) == 180

assert not review_sample.duplicated(
    ['Comment ID', 'Target Label']
).any()

assert review_sample['Case ID'].is_unique

# Verify all sampled cases against saved predictions
for row in review_sample.to_dict('records'):

    i = row['Row Index']
    j = LABELS.index(row['Target Label'])

    assert row['Comment ID'] == test_ids[i]
    assert row['Ground Truth'] == y_test[i, j]
    assert row['Prediction'] == primary_pred[i, j]

    assert row['Ground Truth'] != row['Prediction']

    assert row['Error Type'] == (
        'FP' if row['Ground Truth'] == 0 else 'FN'
    )

# Create blinded manual review worksheet
review_sheet = review_sample[
    ['Case ID', 'Target Label']
].copy()

# Prefix prevents comment text being interpreted as a formula
review_sheet['Comment'] = [
    'TEXT: ' + str(test_df.iloc[i]['comment_text'])
    for i in review_sample['Row Index']
]

review_sheet['Reviewer Judgment'] = ''
review_sheet['Primary Language Category'] = ''
review_sheet['Reviewer Notes'] = ''

# Keep individual comments and predictions private
PRIVATE_DIR.mkdir(parents=True, exist_ok=True)

sample_path = PRIVATE_DIR / 'manual_error_sample.csv'
worksheet_path = PRIVATE_DIR / 'manual_review_worksheet.csv'

# Preserve existing samples and completed annotations
if sample_path.exists():

    existing_sample = pd.read_csv(
        sample_path,
        dtype={'Comment ID': str}
    )

    pd.testing.assert_frame_equal(
        existing_sample,
        review_sample,
        check_dtype=False,
        rtol=1e-12,
        atol=1e-12
    )

else:
    save_csv_atomic(sample_path, review_sample)

if worksheet_path.exists():

    existing_review = pd.read_csv(
        worksheet_path,
        keep_default_na=False
    )

    pd.testing.assert_frame_equal(
        existing_review[
            ['Case ID', 'Target Label', 'Comment']
        ],
        review_sheet[
            ['Case ID', 'Target Label', 'Comment']
        ]
    )

    print('Existing reviewer annotations preserved.')

else:
    save_csv_atomic(worksheet_path, review_sheet)

# Save publication-safe aggregate sampling information
save_csv_atomic(
    TABLES_DIR / 'manual_error_sampling_summary.csv',
    sampling_summary
)

review_sampling_config = {
    'seed': SEED,
    'run_id': PRIMARY_RUN_ID,
    'sample_unit': 'comment-label pair',
    'sampling_method': 'stratified random without replacement',
    'sampled_cases': int(len(review_sample)),
    'unique_comments': int(
        review_sample['Comment ID'].nunique()
    ),
    'sample_sha256': calculate_sha256(sample_path),
    'reviewer_blinded_to_ground_truth': True,
    'reviewer_blinded_to_prediction': True,
    'private_review_artifacts': True,
    'manual_review_completed': False
}

config_path = (
    RUN_CONFIGS_DIR / 'manual_error_sampling_config.json'
)

if config_path.exists():
    assert read_json(config_path) == review_sampling_config
else:
    save_json_atomic(config_path, review_sampling_config)

# Verify the frozen evaluation protocol
assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

print('\nManual error sampling summary')
display(sampling_summary)

print('\nCell 5 summary')
print('Sampled label-level cases:', len(review_sample))
print('Unique comments:', review_sample['Comment ID'].nunique())
print('Sampling seed:', SEED)
print('Sample verification: Passed')
print('Blinded review worksheet:', worksheet_path)
print('Private review files saved: 2')
print('Manual review status: Pending')
print('Additional training and inference: None')
print('Frozen protocol: Unchanged')
print('Cell 5 sampling completed successfully.')

In [ ]:
from google.colab import files

files.download(
    str(PRIVATE_DIR / 'manual_review_worksheet.csv')
)

In [ ]:
from pathlib import Path

matches = list(
    Path('/content/drive/MyDrive').rglob(
        'manual_review_worksheet_AI_assisted*.csv'
    )
)

if matches:
    for path in matches:
        print('Found:', path)
        print('Size:', path.stat().st_size, 'bytes')
else:
    print('File not found in Google Drive.')

### Private annotation reconstruction

The original cell is excluded from public distribution because it contains an encoded private annotation payload. Exact replay requires the separately retained AI-assisted review worksheet. Independent human validation remains pending.

In [ ]:
# Cell 6: Error taxonomy and final verification

from datetime import datetime, timezone
from PIL import Image

# Load the verified AI-assisted review
ai_review_path = PRIVATE_DIR / 'manual_review_worksheet_AI_assisted.csv'

EXPECTED_REVIEW_HASH = (
    '6f1902700155d8ae161486180f417764'
    '23597aa7e40780503b627b6b60e88c06'
)

assert ai_review_path.is_file()

assert calculate_sha256(ai_review_path) == EXPECTED_REVIEW_HASH

review = pd.read_csv(
    ai_review_path,
    dtype=str,
    keep_default_na=False
)

original = pd.read_csv(
    PRIVATE_DIR / 'manual_review_worksheet.csv',
    dtype=str,
    keep_default_na=False
)

sample_path = PRIVATE_DIR / 'manual_error_sample.csv'

sample_config = read_json(
    RUN_CONFIGS_DIR / 'manual_error_sampling_config.json'
)

assert calculate_sha256(sample_path) == (
    sample_config['sample_sha256']
)

sample = pd.read_csv(
    sample_path,
    dtype={'Comment ID': str}
)

# Verify review integrity
identity_columns = ['Case ID', 'Target Label', 'Comment']

pd.testing.assert_frame_equal(
    review[identity_columns],
    original[identity_columns]
)

assert len(review) == len(sample) == 180
assert review['Case ID'].is_unique
assert sample['Case ID'].is_unique

valid_judgments = {'0', '1', 'uncertain'}

valid_categories = {
    'EXPLICIT', 'IMPLICIT', 'SARCASM',
    'QUOTATION', 'PROFANITY', 'IDENTITY_NEUTRAL',
    'CONTEXT', 'AMBIGUOUS', 'OTHER'
}

assert review['Reviewer Judgment'].isin(
    valid_judgments
).all()

assert review['Primary Language Category'].isin(
    valid_categories
).all()

assert review['Reviewer Notes'].str.strip().ne('').all()

print('Review integrity: Verified')

# Unblind the original labels and predictions
review_analysis = sample.merge(
    review,
    on=['Case ID', 'Target Label'],
    validate='one_to_one'
)

assert len(review_analysis) == 180

primary_pred = primary_predictions[
    [f'pred_optimized_{label}' for label in LABELS]
].to_numpy(dtype=bool)

# Verify each sampled error
for row in review_analysis.to_dict('records'):

    i = int(row['Row Index'])
    j = LABELS.index(row['Target Label'])

    assert row['Comment ID'] == test_ids[i]
    assert row['Ground Truth'] == y_test[i, j]
    assert row['Prediction'] == primary_pred[i, j]
    assert row['Ground Truth'] != row['Prediction']

    assert row['Error Type'] == (
        'FP' if row['Ground Truth'] == 0 else 'FN'
    )

    assert row['Comment'] == (
        'TEXT: ' + str(test_df.iloc[i]['comment_text'])
    )

# Compare AI-assisted judgment with dataset and model
def agreement_outcome(row):

    judgment = row['Reviewer Judgment']

    if judgment == 'uncertain':
        return 'Uncertain'

    judgment = int(judgment)

    if judgment == row['Ground Truth']:
        return 'Agrees with Dataset'

    if judgment == row['Prediction']:
        return 'Agrees with Model'

    raise ValueError('Unexpected judgment')


review_analysis['Review Outcome'] = (
    review_analysis.apply(agreement_outcome, axis=1)
)

outcome_order = [
    'Agrees with Dataset',
    'Agrees with Model',
    'Uncertain'
]

# Save individual review cases privately
save_csv_atomic(
    PRIVATE_DIR / 'manual_error_review_analysis.csv',
    review_analysis
)

# Agreement by target label and error type
agreement_summary = pd.crosstab(
    [
        review_analysis['Target Label'],
        review_analysis['Error Type']
    ],
    review_analysis['Review Outcome']
).reindex(
    columns=outcome_order,
    fill_value=0
).reset_index()

agreement_summary['Total Cases'] = (
    agreement_summary[outcome_order].sum(axis=1)
)

for outcome in outcome_order:

    agreement_summary[f'{outcome} (%)'] = (
        100 * agreement_summary[outcome] /
        agreement_summary['Total Cases']
    )

assert len(agreement_summary) == 8
assert agreement_summary['Total Cases'].sum() == 180

# Overall language taxonomy
taxonomy_summary = (
    review_analysis['Primary Language Category']
    .value_counts()
    .rename_axis('Language Category')
    .reset_index(name='Cases')
)

taxonomy_summary['Share of Review Sample (%)'] = (
    100 * taxonomy_summary['Cases'] / len(review_analysis)
)

assert taxonomy_summary['Cases'].sum() == 180

# Language taxonomy by label and error type
taxonomy_by_error = pd.crosstab(
    [
        review_analysis['Target Label'],
        review_analysis['Error Type']
    ],
    review_analysis['Primary Language Category']
).reset_index()

assert taxonomy_by_error.iloc[:, 2:].to_numpy().sum() == 180

# Save aggregate research results
save_csv_atomic(
    TABLES_DIR / 'manual_review_agreement_summary.csv',
    agreement_summary
)

save_csv_atomic(
    TABLES_DIR / 'manual_error_taxonomy_summary.csv',
    taxonomy_summary
)

save_csv_atomic(
    TABLES_DIR / 'manual_taxonomy_by_error_type.csv',
    taxonomy_by_error
)

# Overall agreement results
overall_agreement = (
    review_analysis['Review Outcome']
    .value_counts()
    .reindex(outcome_order, fill_value=0)
)

assert overall_agreement.sum() == 180
assert overall_agreement['Uncertain'] == 32

print('\nOverall AI-assisted review outcomes')
display(overall_agreement.rename('Cases').to_frame())

print('\nAgreement by target label and error type')
display(agreement_summary.round(4))

print('\nLanguage taxonomy')
display(taxonomy_summary.round(4))

print('\nTaxonomy by error type')
display(taxonomy_by_error)

# Visualize agreement and language categories
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

plot_data = agreement_summary.copy()

plot_data['Group'] = (
    plot_data['Target Label'].str.replace('_', ' ') +
    ' — ' + plot_data['Error Type']
)

plot_data.set_index('Group')[outcome_order].plot(
    kind='barh',
    stacked=True,
    color=['#326796', '#E09B45', '#AAB2BD'],
    ax=axes[0]
)

axes[0].set_title('AI-assisted review agreement')
axes[0].set_xlabel('Sampled cases')
axes[0].set_ylabel('')
axes[0].invert_yaxis()
axes[0].legend(fontsize=8)

taxonomy_plot = taxonomy_summary.sort_values('Cases')

axes[1].barh(
    taxonomy_plot['Language Category']
    .str.replace('_', ' ')
    .str.title(),
    taxonomy_plot['Cases'],
    color='#326796'
)

axes[1].set_title('Observed language categories')
axes[1].set_xlabel('Sampled cases')

for ax in axes:
    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)

fig.suptitle(
    'Stratified error review — 180 label-level cases'
)

plt.tight_layout()

fig.savefig(
    FIGURES_DIR / 'F04_manual_error_taxonomy.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

# Verify all Notebook 08 aggregate artifacts
expected_tables = {
    'label_error_summary.csv': 12,
    'paired_model_error_comparison.csv': 12,
    'validation_test_label_comparison.csv': 6,
    'validation_test_macro_comparison.csv': 7,
    'prevalence_gap_decomposition.csv': 12,
    'text_length_error_summary.csv': 12,
    'per_label_length_error_summary.csv': 72,
    'actual_truncation_summary.csv': 2,
    'manual_error_sampling_summary.csv': 8,
    'manual_review_agreement_summary.csv': 8,
    'manual_error_taxonomy_summary.csv': 9,
    'manual_taxonomy_by_error_type.csv': 8
}

artifact_inventory = []

for filename, expected_rows in expected_tables.items():

    path = TABLES_DIR / filename

    assert path.is_file()
    assert len(pd.read_csv(path)) == expected_rows

    artifact_inventory.append({
        'path': str(path.relative_to(PROJECT_ROOT)),
        'sha256': calculate_sha256(path),
        'size_bytes': path.stat().st_size
    })

# Verify saved research figures
expected_figures = [
    'F01_label_error_comparison.png',
    'F02_validation_test_performance_gap.png',
    'F03_text_length_error_analysis.png',
    'F04_manual_error_taxonomy.png'
]

for filename in expected_figures:

    path = FIGURES_DIR / filename

    with Image.open(path) as img:

        assert img.format == 'PNG'

        assert all(
            abs(value - 300) < 1
            for value in img.info['dpi']
        )

        img.verify()

    artifact_inventory.append({
        'path': str(path.relative_to(PROJECT_ROOT)),
        'sha256': calculate_sha256(path),
        'size_bytes': path.stat().st_size
    })

# Verify that the frozen protocol is unchanged
assert calculate_sha256(
    FROZEN_MANIFEST_PATH
) == EXPECTED_FROZEN_HASH

assert calculate_sha256(
    REGISTRY_PATH
) == frozen_manifest['registry']['sha256']

assert calculate_sha256(
    DECISION_LOG_PATH
) == frozen_manifest['decision_log_sha256']

assert calculate_sha256(
    EVALUATION_COMPLETION_PATH
) == error_analysis_setup['evaluation_completion_sha256']

# Record completion and methodological limitations
error_analysis_completion = {

    'notebook': '08_error_analysis',
    'status': 'Completed with human validation pending',

    'primary_run_id': PRIMARY_RUN_ID,
    'baseline_run_id': BASELINE_RUN_ID,

    'validation_samples': len(validation_df),
    'test_samples': len(test_df),

    'review_cases': len(review_analysis),
    'unique_review_comments': int(
        review_analysis['Comment ID'].nunique()
    ),

    'review_seed': SEED,
    'review_method': 'Blinded ChatGPT-assisted linguistic review',
    'ai_assisted_review_completed': True,
    'independent_human_validation_completed': False,
    'review_sha256': EXPECTED_REVIEW_HASH,

    'agreement_counts': {
        name: int(count)
        for name, count in overall_agreement.items()
    },

    'review_limitations': [
        'AI-assisted judgments are not verified ground truth.',
        'Review includes model-error cases only.',
        'Stratified sample is not proportionally representative.',
        'Agreement does not establish annotation correctness.',
        'Original conversational context may be unavailable.',
        'Independent human validation remains pending.'
    ],

    'verified_tables': len(expected_tables),
    'verified_figures': len(expected_figures),
    'artifact_inventory': artifact_inventory,

    'frozen_manifest_sha256': EXPECTED_FROZEN_HASH,
    'thresholds_changed': False,
    'test_driven_selection_performed': False,
    'model_training_performed': False,
    'additional_inference_performed': False,
    'private_review_artifacts': True,

    'next_notebook': '09_fairness_audit',

    'completed_at': datetime.now(
        timezone.utc
    ).isoformat()
}

save_json_atomic(
    LOGS_DIR / 'error_analysis_completion_summary.json',
    error_analysis_completion
)

print('\nNotebook 08 completion summary')

for key, value in error_analysis_completion.items():

    if key not in ['artifact_inventory', 'review_limitations']:
        print(f'{key}: {value}')

print('\nFrozen protocol: Unchanged')
print('Additional training and inference: None')
print('Cell 6 completed successfully.')